# PMM Dynamic Screener — NonKYC Public REST

This notebook screens **NONKYC** markets for **PMM Dynamic (Predictive Market Making)** using only public market-data endpoints. It is a **pre-ingestion gate**: the output is a ranked shortlist plus a candle-ingestor manifest, selected `BASE-QUOTE` pairs, and rule-estimate metadata.

By default this notebook screens **all quote assets** available on NonKYC (USDT, XMR, BTC, USDC, etc.). To restrict to a single quote, set `QUOTE_ASSET` to e.g. `'USDT'`. To screen a specific set, use comma-separated values like `'USDT,XMR'`. The notebook uses the documented public REST endpoints for markets, tickers, order books, candles, and trades.

Stop-ship stance: a pair passing this notebook is **fit for research and ingestion only** until it survives candle-quality checks, walk-forward validation, and live microstructure review.


In [1]:
import os
import sys
import subprocess
import logging
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

CANDIDATE_DIRS = [
    Path("/quants-lab/research_notebooks/market_lab/pmm_dynamic"),
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
]
PMM_DIR = next((p.resolve() for p in CANDIDATE_DIRS if (p / "pyproject.toml").exists() and (p / "pmm_lab").exists()), None)
if PMM_DIR is None:
    raise FileNotFoundError("Could not locate the pmm_dynamic project root. Open this notebook from inside the pmm_dynamic repo.")

if str(PMM_DIR) not in sys.path:
    sys.path.insert(0, str(PMM_DIR))
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-e", str(PMM_DIR), "--quiet"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

logging.getLogger("urllib3").setLevel(logging.WARNING)
logging.getLogger("pmm_lab").setLevel(logging.WARNING)

print(f"PMM project root: {PMM_DIR}")


from pmm_lab.screener import NonKYCPublicScreener, default_nonkyc_config, export_screening_artifacts
from pmm_lab.screener.common import compute_coarse_scores, select_shortlist
from pmm_lab.screener.nonkyc_public import NONKYC_BASE_URL


PMM project root: /quants-lab/research_notebooks/market_lab/pmm_dynamic


## 1. Configuration

Edit the universe, thresholds, and output path here. Keep the defaults conservative unless you explicitly want a wider exploratory net.


In [2]:
# ============================================================
# USER CONFIG — edit these, then Run All
# ============================================================
QUOTE_ASSET = "*"
INTERVAL = "5m"
UNIVERSE_TOP_K = 80
FINAL_TOP_N = 15
CANDLE_LIMIT = 288
DEPTH_LIMIT = 200
RECENT_TRADE_LIMIT = 500

# Optional pair overrides.
# Keep INCLUDE_SYMBOLS empty to use the full eligible universe.
INCLUDE_SYMBOLS = []
EXCLUDE_SYMBOLS = []

OUTPUT_ROOT = (
    PMM_DIR / "artifacts" / "screener" / "nonkyc" /
    datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
)

cfg = default_nonkyc_config()
cfg.quote_asset = QUOTE_ASSET
cfg.interval = INTERVAL
cfg.universe_top_k = UNIVERSE_TOP_K
cfg.final_top_n = FINAL_TOP_N
cfg.candle_limit = CANDLE_LIMIT
cfg.depth_limit = DEPTH_LIMIT
cfg.recent_trade_limit = RECENT_TRADE_LIMIT
cfg.include_symbols = tuple(INCLUDE_SYMBOLS)
cfg.exclude_symbols = tuple(EXCLUDE_SYMBOLS)

# Conservative defaults. Loosen only if you explicitly want more exploratory coverage.
cfg.min_quote_volume_24h = 50000.0
cfg.max_spread_bps = 120.0
cfg.min_top_of_book_quote = 5.0
cfg.min_depth_10bps_quote = 0.0
cfg.max_last_trade_age_sec = 3600.0
cfg.min_recent_trade_count = 40
cfg.min_candle_count = 220
cfg.min_candle_coverage_ratio = 0.9
cfg.max_zero_volume_fraction = 0.3
cfg.min_natr_bps = 12.0
cfg.max_natr_bps = 400.0

print("Screener config")
display(pd.Series(cfg.as_dict()).to_frame("value"))
print(f"Output root: {OUTPUT_ROOT}")


Screener config


,value
connector,nonkyc
quote_asset,*
interval,5m
universe_top_k,80
final_top_n,15
candle_limit,288
depth_limit,200
recent_trade_limit,500
request_pause_sec,0.2
timeout_seconds,30.0


Output root: /quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/nonkyc/20260325_235054


## 2. Build the cheap universe snapshot

This step uses only batch endpoints to discover the broad market universe and assign a coarse score before any per-pair enrichment calls are made.


In [3]:
screener = NonKYCPublicScreener(cfg)

universe = screener.build_universe()
universe = compute_coarse_scores(universe)
shortlist = select_shortlist(universe, cfg)

summary = pd.DataFrame([
    {
        "connector": cfg.connector,
        "quote_asset": cfg.quote_asset,
        "interval": cfg.interval,
        "universe_rows": len(universe),
        "shortlist_rows": len(shortlist),
    }
])
display(summary)

preview_cols = [
    c for c in [
        "trading_pair",
        "exchange_symbol",
        "quote_volume_24h",
        "spread_bps",
        "coarse_score",
        "price_tick_estimate",
        "amount_step_estimate",
        "status_detail",
    ]
    if c in universe.columns
]

display(universe.loc[:, preview_cols].head(20))
print(f"Shortlist for detailed enrichment: {len(shortlist)}")
display(shortlist.loc[:, preview_cols].head(min(25, len(shortlist))))


,connector,quote_asset,interval,universe_rows,shortlist_rows
0,nonkyc,*,5m,345,80


,trading_pair,exchange_symbol,quote_volume_24h,spread_bps,coarse_score,price_tick_estimate,amount_step_estimate,status_detail
0,NKYC-USDT,NKYC/USDT,1.685141e+05,10.514318,76.545639,1.000000e-06,0.000100,active
1,SUN-USDT,SUN/USDT,1.458112e+05,11.180055,76.080600,1.000000e-06,0.010000,active
2,ARRR-USDT,ARRR/USDT,2.814920e+05,24.737313,75.708111,1.000000e-06,0.000100,active
3,LTC-USDT,LTC/USDT,8.153326e+05,40.848948,75.435164,1.000000e-02,0.000100,active
4,XRP-USDT,XRP/USDT,2.103925e+06,54.607993,75.075896,1.000000e-04,0.010000,active
5,BTC-USDC,BTC/USDC,7.075347e+05,43.311014,74.983346,1.000000e-02,0.000001,active
6,USDC-USDT,USDC/USDT,1.683752e+06,51.994801,74.816934,1.000000e-04,0.010000,active
7,SOL-USDT,SOL/USDT,2.026641e+06,61.208875,74.404780,1.000000e-02,0.010000,active
8,LINK-USDT,LINK/USDT,1.074873e+06,53.734551,74.351894,1.000000e-02,0.010000,active
9,ETH-USDT,ETH/USDT,7.457523e+06,65.660003,73.627129,1.000000e-02,0.000010,active


Shortlist for detailed enrichment: 80


,trading_pair,exchange_symbol,quote_volume_24h,spread_bps,coarse_score,price_tick_estimate,amount_step_estimate,status_detail
0,NKYC-USDT,NKYC/USDT,1.685141e+05,10.514318,76.545639,1.000000e-06,0.000100,active
1,SUN-USDT,SUN/USDT,1.458112e+05,11.180055,76.080600,1.000000e-06,0.010000,active
2,ARRR-USDT,ARRR/USDT,2.814920e+05,24.737313,75.708111,1.000000e-06,0.000100,active
3,LTC-USDT,LTC/USDT,8.153326e+05,40.848948,75.435164,1.000000e-02,0.000100,active
4,XRP-USDT,XRP/USDT,2.103925e+06,54.607993,75.075896,1.000000e-04,0.010000,active
5,BTC-USDC,BTC/USDC,7.075347e+05,43.311014,74.983346,1.000000e-02,0.000001,active
6,USDC-USDT,USDC/USDT,1.683752e+06,51.994801,74.816934,1.000000e-04,0.010000,active
7,SOL-USDT,SOL/USDT,2.026641e+06,61.208875,74.404780,1.000000e-02,0.010000,active
8,LINK-USDT,LINK/USDT,1.074873e+06,53.734551,74.351894,1.000000e-02,0.010000,active
9,ETH-USDT,ETH/USDT,7.457523e+06,65.660003,73.627129,1.000000e-02,0.000010,active


## 3. Run the detailed screener

This step enriches the shortlist with order-book depth, recent trades, and candles, then applies hard gates plus the final PMM-oriented score.


In [4]:
run = screener.screen_from_universe(universe)
final_df = run.final.copy()
selected_df = run.selected.copy()

status_counts = pd.DataFrame([
    {
        "enriched_rows": len(final_df),
        "selected_rows": len(selected_df),
        "pass_rate": float(selected_df.shape[0] / final_df.shape[0]) if len(final_df) else 0.0,
    }
])
display(status_counts)

final_cols = [
    c for c in [
        "trading_pair",
        "screen_score",
        "passed_filters",
        "quote_volume_24h",
        "spread_bps",
        "top_of_book_quote",
        "sym_depth_quote_10bps",
        "recent_trade_count",
        "last_trade_age_sec",
        "n_candles",
        "coverage_ratio",
        "zero_volume_fraction",
        "natr_bps_mean",
        "efficiency_ratio",
        "rejection_reason",
    ]
    if c in final_df.columns
]
display(final_df.loc[:, final_cols].head(min(50, len(final_df))))


GET https://api.nonkyc.io/api/v2/market/orderbook?symbol=MANA%2FUSDT&limit=200 failed on attempt 1/3: HTTP Error 502: Bad Gateway
GET https://api.nonkyc.io/api/v2/market/orderbook?symbol=MORPHO%2FUSDT&limit=200 failed on attempt 1/3: HTTP Error 502: Bad Gateway
GET https://api.nonkyc.io/api/v2/market/trades?symbol=DIVI%2FUSDT&limit=500 failed on attempt 1/3: HTTP Error 502: Bad Gateway
GET https://api.nonkyc.io/api/v2/market/orderbook?symbol=EIGEN%2FUSDT&limit=200 failed on attempt 1/3: HTTP Error 502: Bad Gateway
GET https://api.nonkyc.io/api/v2/market/orderbook?symbol=ERG%2FUSDT&limit=200 failed on attempt 1/3: HTTP Error 502: Bad Gateway
GET https://api.nonkyc.io/api/v2/market/trades?symbol=ERG%2FUSDT&limit=500 failed on attempt 1/3: HTTP Error 502: Bad Gateway


,enriched_rows,selected_rows,pass_rate
0,80,10,0.125


,trading_pair,screen_score,passed_filters,quote_volume_24h,spread_bps,top_of_book_quote,sym_depth_quote_10bps,recent_trade_count,last_trade_age_sec,n_candles,coverage_ratio,zero_volume_fraction,natr_bps_mean,efficiency_ratio,rejection_reason
0,USDC-USDT,82.172668,True,1.683752e+06,51.994801,472.973590,0.000000,200,0.459929,288,1.000000,0.000000,15.033446,0.001821,
1,SOL-USDT,78.261385,True,2.026641e+06,61.208875,1596.798000,0.000000,200,7.193323,288,1.000000,0.000000,27.132022,0.027983,
2,UNI-USDT,74.166667,True,3.632007e+05,78.346616,2204.442680,0.000000,200,21.495398,288,1.000000,0.000000,26.936767,0.063014,
3,LTC-USDT,72.022096,True,8.153326e+05,40.848948,28.238210,0.000000,200,9.173946,288,1.000000,0.000000,22.615545,0.004969,
4,AVAX-USDT,71.221316,True,5.110441e+05,82.730093,258.771500,0.000000,200,12.886237,288,0.989691,0.000000,23.203397,0.031008,
5,AAVE-USDT,70.953422,True,1.001274e+05,65.736875,369.313800,0.000000,200,2.164241,288,1.000000,0.000000,26.030989,0.016089,
6,RENDER-USDT,69.469167,True,1.126968e+05,75.471698,887.040000,0.000000,200,12.215571,288,1.000000,0.000000,40.651349,0.086482,
7,NKYC-USDT,69.279822,True,1.685141e+05,10.514318,36.864207,36.864207,200,28.938642,288,1.000000,0.000000,13.821795,0.004851,
8,PEPE-USDT,67.895280,True,1.196857e+05,56.657224,119.553022,0.000000,200,18.749890,288,0.911392,0.000000,13.046437,0.063291,
9,BCH-USDT,64.723030,True,1.328396e+05,65.352588,5.664144,0.000000,200,8.646626,288,1.000000,0.003472,27.252494,0.012567,


## 4. Diagnostics

Inspect which pairs passed, which pairs failed, and why. Rejection counts are often more useful than raw rankings because they show whether you are liquidity-bound, spread-bound, or data-quality-bound.


In [5]:
if final_df.empty:
    print("No enriched rows returned.")
else:
    passed = final_df[final_df["passed_filters"]].copy()
    rejected = final_df[~final_df["passed_filters"]].copy()

    print(f"Passed: {{len(passed)}} | Rejected: {{len(rejected)}}")

    if not passed.empty:
        display(
            passed.loc[:, [
                c for c in [
                    "trading_pair",
                    "screen_score",
                    "quote_volume_24h",
                    "spread_bps",
                    "top_of_book_quote",
                    "sym_depth_quote_10bps",
                    "recent_trade_count",
                    "last_trade_age_sec",
                    "natr_bps_mean",
                    "efficiency_ratio",
                ] if c in passed.columns
            ]].head(cfg.final_top_n)
        )

    if not rejected.empty:
        reject_counts = (
            rejected["rejection_reason"]
            .fillna("")
            .str.split("; ")
            .explode()
            .loc[lambda s: s.ne("")]
            .value_counts()
            .rename_axis("rejection_reason")
            .to_frame("count")
        )
        display(reject_counts.head(20))


Passed: {len(passed)} | Rejected: {len(rejected)}


,trading_pair,screen_score,quote_volume_24h,spread_bps,top_of_book_quote,sym_depth_quote_10bps,recent_trade_count,last_trade_age_sec,natr_bps_mean,efficiency_ratio
0,USDC-USDT,82.172668,1.683752e+06,51.994801,472.973590,0.000000,200,0.459929,15.033446,0.001821
1,SOL-USDT,78.261385,2.026641e+06,61.208875,1596.798000,0.000000,200,7.193323,27.132022,0.027983
2,UNI-USDT,74.166667,3.632007e+05,78.346616,2204.442680,0.000000,200,21.495398,26.936767,0.063014
3,LTC-USDT,72.022096,8.153326e+05,40.848948,28.238210,0.000000,200,9.173946,22.615545,0.004969
4,AVAX-USDT,71.221316,5.110441e+05,82.730093,258.771500,0.000000,200,12.886237,23.203397,0.031008
5,AAVE-USDT,70.953422,1.001274e+05,65.736875,369.313800,0.000000,200,2.164241,26.030989,0.016089
6,RENDER-USDT,69.469167,1.126968e+05,75.471698,887.040000,0.000000,200,12.215571,40.651349,0.086482
7,NKYC-USDT,69.279822,1.685141e+05,10.514318,36.864207,36.864207,200,28.938642,13.821795,0.004851
8,PEPE-USDT,67.895280,1.196857e+05,56.657224,119.553022,0.000000,200,18.749890,13.046437,0.063291
9,BCH-USDT,64.723030,1.328396e+05,65.352588,5.664144,0.000000,200,8.646626,27.252494,0.012567


,count
rejection_reason,
top_of_book_quote<5,66
quote_volume_24h<50000,44
coverage_ratio<0.90,7
natr_bps_mean<12,4


## 5. Export artifacts

This writes CSV, JSON, a Markdown report, a candle-ingestor manifest, and an exchange-rules patch for the selected pairs.


In [6]:
artifact_paths = export_screening_artifacts(
    run,
    output_dir=str(OUTPUT_ROOT),
    base_url=NONKYC_BASE_URL,
    trade_limit=cfg.recent_trade_limit,
)

print("Artifacts written")
display(pd.Series(asdict(artifact_paths)).to_frame("path"))


Artifacts written


,path
root_dir,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/nonkyc/20260325_235054
universe_csv,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/nonkyc/20260325_235054/universe.csv
shortlist_csv,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/nonkyc/20260325_235054/shortlist.csv
final_csv,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/nonkyc/20260325_235054/final_screen.csv
selected_csv,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/nonkyc/20260325_235054/selected_pairs.csv
selected_pairs_txt,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/nonkyc/20260325_235054/selected_pairs.txt
selected_pairs_json,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/nonkyc/20260325_235054/selected_pairs.json
symbol_metadata_json,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/nonkyc/20260325_235054/symbol_metadata.json
candle_ingestor_manifest_yaml,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/nonkyc/20260325_235054/candle_ingestor_mani...
exchange_rules_patch_yaml,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/nonkyc/20260325_235054/exchange_rules_patch...


## 6. Merge into candle ingestion

The manifest below follows the same `exchanges -> pairs -> intervals` structure already used by your candle-ingest and candle-gap-repair tooling, with slash-formatted exchange pairs and Hummingbot-normalized selected pair files alongside it.


In [7]:
from pathlib import Path

print("Selected Hummingbot pairs")
print("-" * 80)
print(Path(artifact_paths.selected_pairs_txt).read_text())

print("\nCandle ingestor manifest")
print("-" * 80)
print(Path(artifact_paths.candle_ingestor_manifest_yaml).read_text())

print("\nExchange rules patch (estimates only)")
print("-" * 80)
print(Path(artifact_paths.exchange_rules_patch_yaml).read_text())


Selected Hummingbot pairs
--------------------------------------------------------------------------------
USDC-USDT
SOL-USDT
UNI-USDT
LTC-USDT
AVAX-USDT
AAVE-USDT
RENDER-USDT
NKYC-USDT
PEPE-USDT
BCH-USDT


Candle ingestor manifest
--------------------------------------------------------------------------------
backfill_days: 180
request_delay: 0.5
overlap_candles: 2
include_open_candle: false
http:
  timeout_seconds: 30.0
  max_retries: 3
  retry_backoff: 1.8
  user_agent: pmm-lab-screener/0.1
exchanges:
  nonkyc:
    enabled: true
    base_url: https://api.nonkyc.io/api/v2
    pairs:
    - USDC/USDT
    - SOL/USDT
    - UNI/USDT
    - LTC/USDT
    - AVAX/USDT
    - AAVE/USDT
    - RENDER/USDT
    - NKYC/USDT
    - PEPE/USDT
    - BCH/USDT
    intervals:
    - 5m
    trades:
      enabled: true
      limit: 500
      update_recent_candles: false
      recent_window_minutes: 120


Exchange rules patch (estimates only)
--------------------------------------------------------------------